# Module 28 — Reflexion and Self-Refine

**THE ONE IDEA:** the agent failed. It writes down **why**, stores that, and tries again.
No gradients, no fine-tuning — **the "learning" is the written reflection**, and it lives
in module 25's procedural memory.

Two related patterns, and the difference matters:

| | feedback from | learns across |
|---|---|---|
| **Self-Refine** | the model critiquing its own output | **one task** — gone afterwards |
| **Reflexion** | an *external* signal (a test, a checker, a failure) | **future tasks** — it persists |

And the thing that stops either running forever: **three stopping criteria**, all of them
implemented here as an OR.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import math
from _providers import get_client
from _tools import run_tool

client, MODEL, _ = get_client("openai")
REFLECTIONS: list[str] = []           # module 25's procedural store, in miniature

def call(prompt, max_tok=350, system=None):
    msgs = ([{"role": "system", "content": system}] if system else [])
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, max_tokens=max_tok, messages=msgs)
    return r.choices[0].message.content.strip()

# An EXTERNAL verifier. This is what separates Reflexion from self-flattery.
def verify(answer, expected="10000"):
    ok = expected in answer.replace(",", "").replace(" ", "")
    return ok, ("correct" if ok else f"WRONG — expected {expected} in the answer")

## Reflexion — attempt, verify, reflect, retry

The task is deliberately phrased so a first attempt tends to reach for the wrong policy
keyword.

In [ ]:
TASK = ("Using search_policy and the ERC schedule, what is the year-2 early "
        "repayment charge on a 250000 loan? Give the number.")

def attempt(n):
    memo = ("\n".join(f"- {r}" for r in REFLECTIONS)) or "(none yet)"
    plan = call(f"PAST LESSONS:\n{memo}\n\nTASK: {TASK}\n"
                "Reply with ONE line: the exact search_policy keyword to use.", max_tok=40)
    kw = plan.strip().strip('"').split()[-1].lower()
    policy = run_tool("search_policy", {"query": kw})
    ans = call(f"POLICY: {policy}\nTASK: {TASK}\nCompute it. State the number.")
    ok, why = verify(ans)
    print(f"  attempt {n}: keyword={kw!r} -> {policy[:44]}")
    print(f"             verdict={why}")
    return ok, kw, policy, ans

for n in (1, 2, 3):
    ok, kw, policy, ans = attempt(n)
    if ok:
        print(f"\nSUCCEEDED on attempt {n}: {ans[:100]}"); break
    refl = call(f"You failed this task.\nTASK: {TASK}\nYou searched for {kw!r} and got "
                f"{policy!r}\nYour answer: {ans[:200]}\nVerdict: {why}\n\n"
                "Write ONE short lesson for your next attempt. Be specific about the keyword.",
                max_tok=90)
    REFLECTIONS.append(refl)
    print(f"             reflection stored: {refl[:88]}\n")

## The reflection persists — a *new* task benefits

In [ ]:
print("stored reflections:")
for r in REFLECTIONS: print("  -", r[:96])
print("\n^ these are procedural memory (module 25). A future run starts with them")
print("  in context and does not have to rediscover the same mistake.")

## Self-Refine, and the three stopping criteria

In [ ]:
def cosine_ish(a, b):
    """Cheap convergence proxy — token overlap. Real systems embed and use cosine."""
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / max(len(sa | sb), 1)

draft, MAX_ROUNDS, hist = call("Write a 3-sentence letter declining a mortgage "
                               "application on affordability grounds."), 4, []
stop = None
for rnd in range(1, MAX_ROUNDS + 1):
    crit = call(f"Critique for empathy, clarity and FCA tone. If it is good enough, "
                f"reply with exactly APPROVED.\n\n{draft}", max_tok=160)
    if "APPROVED" in crit.upper():
        stop = f"criterion 1 — critic signalled done (round {rnd})"; break
    new = call(f"Rewrite addressing this critique.\nCRITIQUE: {crit}\nLETTER: {draft}")
    sim = cosine_ish(draft, new); hist.append(sim)
    print(f"  round {rnd}: similarity to previous = {sim:.2f}")
    draft = new
    if sim > 0.92:
        stop = f"criterion 2 — converged, similarity {sim:.2f} (round {rnd})"; break
else:
    stop = f"criterion 3 — hard cap of {MAX_ROUNDS} rounds"

print(f"\nstopped by: {stop}")
print(f"\nFINAL:\n{draft[:280]}")

print("""
LESSON - two patterns that look alike and are not.

  SELF-REFINE   the model critiques ITSELF. No external signal, so it can be
                confidently wrong in a loop, and the critique is discarded when
                the task ends. Good for QUALITY on generation tasks - tone,
                completeness, structure. Useless for CORRECTNESS.

  REFLEXION     the feedback comes from OUTSIDE - a test suite, a verifier, a
                failed tool call. The reflection is WRITTEN DOWN and persists, so
                attempt 3 of a future task starts smarter. That is verbal
                reinforcement learning: no gradients, just text in a store.

Three stopping criteria, and you implement ALL THREE as an OR, because each one
fails on its own:
  1. the critic says APPROVED        - may never fire if the critic is strict
  2. successive outputs CONVERGE     - catches a critic that nitpicks forever
  3. a hard round cap                - catches everything else

Two honest warnings. A reflection can be WRONG: the model may misdiagnose its own
failure and store a lesson that makes things worse - which is why they need the
same reliability scoring as module 25's skills. And the store grows, so it needs
the same decay and eviction.

Reflexion only works when you have a real verifier. If you cannot tell success
from failure automatically, you cannot reflect on it - you can only self-flatter.""")

---

**Next:** Block J — `../J_multi_agent/29_multi_agent_pipeline.ipynb`